# Baseline greedy (2-gen GS-Sub) over the 640 solved MS(1190)

Classical 2-generator control for the `z=w` (n=3) stabilization arms. Runs
`run_baseline_greedy.py` on `data/1190MS.txt` idx `[START, END)` (the AC-trivial solved set)
at `BUDGET` nodes, per-relator cap `MAX_LEN=24` (the **same cap** the z=w arms use).

Output schema is **identical** to the z=w calibration streams (`arm="baseline"`), so the website
viewer merges baseline vs z=w directly on `dataset|idx|arm|budget_nodes`.

**Run order:** CONFIG → SETUP → RUN. The run is append-only + resumable — safe to re-run after a
Colab disconnect (it skips idx already recorded). Results land in Drive; copy them back into the
repo at `results/baseline_greedy/`.

> Note: the 6 boundary cases (idx 634–639) may exhaust `BUDGET` without solving under the fair
> per-relator cap — the paper cracks those only at higher budget / its sum-cap. They are recorded
> as `solved:false, exhausted_budget:true`; each burns the full budget (~8 min single-core), so
> they dominate wall-time. That's expected, not a bug.

In [ ]:
# ==================== BASELINE GREEDY — CONFIG (edit ONLY this cell) ====================
REPO_URL      = "https://github.com/Avi161/ACSolverX.git"
BRANCH        = "test/stable-ac-moves"
REPO_DIR      = "ACSolverX"
CLONE         = True
UPDATE_REPO   = True          # git reset --hard origin/BRANCH so a RESTART pulls the latest push (needs >= fdd5175)
MOUNT_DRIVE   = True
DRIVE_OUT_DIR = "/content/drive/MyDrive/baseline_greedy"   # results land here; copy back into repo results/baseline_greedy/

START   = 0
END     = 640                 # exclusive; 640 = the AC-trivial solved set (idx 0-639)
BUDGET  = 1_000_000           # node budget per presentation
MAX_LEN = 24                  # per-relator length cap (match the z=w arms)
WORKERS = None                # None = auto (cpu_count-1); pin an int (e.g. 6); 1 = serial
print("config loaded.")

In [ ]:
# ==================== SETUP (clone / pull / install / mount) ====================
import os, sys, subprocess
def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")   # <-- confirm the commit (needs the baseline runner, >= fdd5175)
    sh("pip -q install numba numpy")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    REPO_ROOT = os.path.abspath(REPO_DIR)
else:
    REPO_ROOT = os.path.abspath(os.getcwd())
    while REPO_ROOT != "/" and not os.path.isdir(os.path.join(REPO_ROOT, "envs")):
        REPO_ROOT = os.path.dirname(REPO_ROOT)
print("repo:", REPO_ROOT)

In [ ]:
# ==================== RUN (streams live; resumable — re-run safely after a disconnect) ====================
script  = os.path.join(REPO_ROOT, "experiments", "stable_ac", "one_generator", "run_baseline_greedy.py")
out_dir = DRIVE_OUT_DIR if (IN_COLAB and MOUNT_DRIVE) else os.path.join(REPO_ROOT, "results", "baseline_greedy")
cmd = [sys.executable, script, "--start", str(START), "--end", str(END),
       "--budget", str(BUDGET), "--max_len", str(MAX_LEN), "--out_dir", out_dir]
if WORKERS:
    cmd += ["--workers", str(WORKERS)]
print(" ".join(cmd), "\n")
subprocess.run(cmd, check=True)
print("\n->", out_dir, "(solved/calibration_baseline.jsonl  +  paths/paths_baseline.jsonl)")